# Audio processing consistency audit

This notebook mirrors `get_experiment_month` from `average_audio_files.ipynb` and checks three things for every supported experiment found in raw data:

1. The experiment has a folder in `Processed_data/Audio/<month>/<exp>`.
2. The averaged audio folder has the expected files based on the raw source-channel pairs for that experiment's naming scheme.
3. The raw experiment log text file also appears in the processed audio folder.


In [ ]:
import platform
import re
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
if (REPO_ROOT / "vocalization_analysis").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vocalization_analysis.audio_processing_config import (
    detect_raw_naming_scheme,
    get_channel_mapping,
    get_experiment_month,
    should_skip_experiment,
)

if platform.system() == "Windows":
    base_raw = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data")
    base_processed = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data")
else:
    base_raw = Path("/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/")
    base_processed = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data")

AUDIO_ROOT = base_processed / "Audio"
AUDIO_SUBFOLDER = "Averaged_wavs_w_annotations"
VIRTUAL_CHANNELS = ("10", "20", "30")

print("Raw data base:", base_raw)
print("Processed data base:", base_processed)
print("Audio root:", AUDIO_ROOT)


ModuleNotFoundError: No module named 'vocalization_analysis'

In [ ]:
RAW_CHUNK_PATTERN = re.compile(r"channel_00_file_(\d+)\.wav$")
LEGACY_RAW_PATTERN_TEMPLATE = r"channel_{channel}_(\d+)\.wav$"
PROCESSED_PATTERN_TEMPLATE = "channel_{channel}_file_*.wav"


def supported_experiments(raw_root):
    experiments = []
    skipped = []
    skipped_by_design = []

    for experiment_root in sorted(raw_root.glob("experiment_*")):
        if not experiment_root.is_dir():
            continue

        try:
            exp = int(experiment_root.name.split("_")[1])
        except (IndexError, ValueError):
            skipped.append(experiment_root.name)
            continue

        try:
            month_folder = get_experiment_month(exp)
        except ValueError:
            continue

        if should_skip_experiment(exp):
            skipped_by_design.append(exp)
            continue

        experiments.append((exp, month_folder, experiment_root))

    return experiments, skipped, skipped_by_design


def extract_chunk_ids(folder, glob_pattern, regex=None):
    chunk_ids = set()
    for path in folder.glob(glob_pattern):
        if not path.is_file():
            continue
        if regex is None:
            chunk_ids.add(path.stem)
            continue
        match = regex.match(path.name)
        if match:
            chunk_ids.add(int(match.group(1)))
    return chunk_ids


def collect_raw_chunk_ids(raw_audio_folder, exp):
    raw_layout = detect_raw_naming_scheme(exp, raw_audio_folder)
    channel_mapping = get_channel_mapping(exp)
    raw_chunk_ids_by_virtual_channel = {channel: set() for channel in VIRTUAL_CHANNELS}

    if raw_layout == "modern":
        for virtual_channel, (ch1, ch2) in channel_mapping.items():
            pattern1 = re.compile(rf"channel_{ch1:02d}_file_(\d+)\.wav$")
            pattern2 = re.compile(rf"channel_{ch2:02d}_file_(\d+)\.wav$")
            ids1 = extract_chunk_ids(raw_audio_folder, f"channel_{ch1:02d}_file_*.wav", pattern1)
            ids2 = extract_chunk_ids(raw_audio_folder, f"channel_{ch2:02d}_file_*.wav", pattern2)
            raw_chunk_ids_by_virtual_channel[virtual_channel] = ids1 & ids2
        return raw_layout, raw_chunk_ids_by_virtual_channel

    if raw_layout == "legacy":
        for virtual_channel, (ch1, ch2) in channel_mapping.items():
            pattern1 = re.compile(LEGACY_RAW_PATTERN_TEMPLATE.format(channel=ch1))
            pattern2 = re.compile(LEGACY_RAW_PATTERN_TEMPLATE.format(channel=ch2))
            ids1 = extract_chunk_ids(raw_audio_folder, f"channel_{ch1}_*.wav", pattern1)
            ids2 = extract_chunk_ids(raw_audio_folder, f"channel_{ch2}_*.wav", pattern2)
            raw_chunk_ids_by_virtual_channel[virtual_channel] = ids1 & ids2
        return raw_layout, raw_chunk_ids_by_virtual_channel

    return raw_layout, raw_chunk_ids_by_virtual_channel


def collect_processed_chunk_ids(averaged_folder):
    chunk_ids_by_channel = {}
    for channel in VIRTUAL_CHANNELS:
        pattern = re.compile(rf"channel_{channel}_file_(\d+)\.wav$")
        chunk_ids_by_channel[channel] = extract_chunk_ids(
            averaged_folder,
            PROCESSED_PATTERN_TEMPLATE.format(channel=channel),
            pattern,
        )
    return chunk_ids_by_channel


def audit_experiment(exp, month_folder, experiment_root):
    raw_audio_folder = experiment_root / "concatenated_data_cam_mic_sync"
    processed_audio_folder = AUDIO_ROOT / month_folder / str(exp)
    averaged_folder = processed_audio_folder / AUDIO_SUBFOLDER

    folder_exists = processed_audio_folder.exists()
    averaged_folder_exists = averaged_folder.exists()
    raw_audio_exists = raw_audio_folder.exists()

    raw_layout = "missing"
    raw_chunk_ids_by_virtual_channel = {channel: set() for channel in VIRTUAL_CHANNELS}
    if raw_audio_exists:
        raw_layout, raw_chunk_ids_by_virtual_channel = collect_raw_chunk_ids(raw_audio_folder, exp)

    expected_chunks_ch10 = len(raw_chunk_ids_by_virtual_channel["10"])
    expected_chunks_ch20 = len(raw_chunk_ids_by_virtual_channel["20"])
    expected_chunks_ch30 = len(raw_chunk_ids_by_virtual_channel["30"])
    expected_chunks = max(expected_chunks_ch10, expected_chunks_ch20, expected_chunks_ch30)
    expected_averaged_files = expected_chunks_ch10 + expected_chunks_ch20 + expected_chunks_ch30

    processed_chunk_ids = {channel: set() for channel in VIRTUAL_CHANNELS}
    if averaged_folder_exists:
        processed_chunk_ids = collect_processed_chunk_ids(averaged_folder)

    actual_averaged_files = sum(len(chunk_ids) for chunk_ids in processed_chunk_ids.values())
    missing_by_channel = {
        channel: sorted(raw_chunk_ids_by_virtual_channel[channel] - processed_chunk_ids[channel])
        for channel in VIRTUAL_CHANNELS
    }
    extra_by_channel = {
        channel: sorted(processed_chunk_ids[channel] - raw_chunk_ids_by_virtual_channel[channel])
        for channel in VIRTUAL_CHANNELS
    }

    raw_log_files = sorted(experiment_root.glob(f"experiment_{exp}_log*.txt"))
    processed_log_files = []
    if folder_exists:
        processed_log_files = sorted(processed_audio_folder.glob(f"experiment_{exp}_log*.txt"))

    missing_log_files = [path.name for path in raw_log_files if not (processed_audio_folder / path.name).exists()]

    return {
        "exp": exp,
        "month_folder": month_folder,
        "raw_layout": raw_layout,
        "raw_audio_exists": raw_audio_exists,
        "processed_folder_exists": folder_exists,
        "averaged_folder_exists": averaged_folder_exists,
        "raw_chunk_count": expected_chunks,
        "expected_chunks_ch10": expected_chunks_ch10,
        "expected_chunks_ch20": expected_chunks_ch20,
        "expected_chunks_ch30": expected_chunks_ch30,
        "expected_averaged_file_count": expected_averaged_files,
        "actual_averaged_file_count": actual_averaged_files,
        "file_count_matches": expected_averaged_files == actual_averaged_files,
        "raw_log_count": len(raw_log_files),
        "processed_log_count": len(processed_log_files),
        "all_raw_logs_present_in_processed": len(raw_log_files) > 0 and not missing_log_files,
        "missing_processed_chunks_ch10": missing_by_channel["10"],
        "missing_processed_chunks_ch20": missing_by_channel["20"],
        "missing_processed_chunks_ch30": missing_by_channel["30"],
        "extra_processed_chunks_ch10": extra_by_channel["10"],
        "extra_processed_chunks_ch20": extra_by_channel["20"],
        "extra_processed_chunks_ch30": extra_by_channel["30"],
        "missing_log_files": missing_log_files,
        "raw_audio_folder": str(raw_audio_folder),
        "processed_audio_folder": str(processed_audio_folder),
        "averaged_folder": str(averaged_folder),
    }


In [ ]:
experiments, skipped_names, skipped_by_design = supported_experiments(base_raw)
results = []

total_experiments = len(experiments)
for idx, (exp, month_folder, experiment_root) in enumerate(experiments, start=1):
    print(f"Auditing exp {exp} ({idx}/{total_experiments})")
    result = audit_experiment(exp, month_folder, experiment_root)
    results.append(result)

    status_parts = []
    if not result["processed_folder_exists"]:
        status_parts.append("missing processed folder")
    if not result["file_count_matches"]:
        status_parts.append(
            f"file count mismatch {result['actual_averaged_file_count']}/{result['expected_averaged_file_count']}"
        )
    if not result["all_raw_logs_present_in_processed"]:
        status_parts.append("missing processed log copy")

    if status_parts:
        print("  " + "; ".join(status_parts))
    else:
        print("  ok")

audit_df = pd.DataFrame(results).sort_values("exp").reset_index(drop=True)

print(f"Supported experiments found in raw data: {len(audit_df)}")
if skipped_names:
    print(f"Skipped non-standard entries: {skipped_names}")
if skipped_by_design:
    print(f"Skipped by design: {skipped_by_design}")

missing_audio_folders = audit_df.loc[~audit_df["processed_folder_exists"], ["exp", "month_folder", "processed_audio_folder"]]
print("\nExperiments missing Processed_data/Audio folders:")
if missing_audio_folders.empty:
    print("None")
else:
    print(missing_audio_folders.to_string(index=False))

file_count_mismatches = audit_df.loc[
    ~audit_df["file_count_matches"],
    [
        "exp",
        "month_folder",
        "raw_layout",
        "raw_chunk_count",
        "expected_chunks_ch10",
        "expected_chunks_ch20",
        "expected_chunks_ch30",
        "expected_averaged_file_count",
        "actual_averaged_file_count",
        "missing_processed_chunks_ch10",
        "missing_processed_chunks_ch20",
        "missing_processed_chunks_ch30",
        "extra_processed_chunks_ch10",
        "extra_processed_chunks_ch20",
        "extra_processed_chunks_ch30",
    ],
]
print("\nExperiments with averaged-file count mismatches:")
if file_count_mismatches.empty:
    print("None")
else:
    print(file_count_mismatches.to_string(index=False))

missing_logs = audit_df.loc[
    ~audit_df["all_raw_logs_present_in_processed"],
    ["exp", "month_folder", "raw_log_count", "processed_log_count", "missing_log_files"],
]
print("\nExperiments missing copied raw log text files in Audio:")
if missing_logs.empty:
    print("None")
else:
    print(missing_logs.to_string(index=False))

audit_df.head()


In [ ]:
summary = pd.Series(
    {
        "supported_experiments": len(audit_df),
        "missing_processed_audio_folders": int((~audit_df["processed_folder_exists"]).sum()),
        "averaged_file_count_mismatches": int((~audit_df["file_count_matches"]).sum()),
        "missing_or_incomplete_log_copies": int((~audit_df["all_raw_logs_present_in_processed"]).sum()),
    }
)
summary
